In [ ]:
# --- Imports ---
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import torch.nn.functional as F
import numpy as np
import pickle
from tqdm import tqdm
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ---------------------------------------------------
# 1. Define Hybrid Encoder (same as in hybrid training)
# ---------------------------------------------------
class HybridEncoder(nn.Module):
    def __init__(self, out_dim=1024, proj_dim=128):
        super(HybridEncoder, self).__init__()
        base = models.resnet18(weights=None)
        self.encoder = nn.Sequential(*list(base.children())[:-1])
        in_features = base.fc.in_features
        self.projection = nn.Sequential(
            nn.Linear(in_features, out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, proj_dim)
        )

    def forward(self, x):
        x = self.encoder(x)
        x = torch.flatten(x, 1)
        z = self.projection(x)
        return F.normalize(z, dim=1)

# ---------------------------------------------------
# 2. Load pretrained Hybrid Encoder
# ---------------------------------------------------
model = HybridEncoder(out_dim=1024, proj_dim=128)
state_dict = torch.load("hybrid_encoder.pth", map_location=device)
missing, unexpected = model.load_state_dict(state_dict, strict=False)
print(f"Missing keys: {missing}\nUnexpected keys: {unexpected}")

model = model.to(device)
print("✅ Hybrid encoder loaded and ready for fine-tuning.")

# ---------------------------------------------------
# 3. Freeze most layers (optional)
# ---------------------------------------------------
for name, param in model.encoder.named_parameters():
    if "layer4" in name or "fc" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

# ---------------------------------------------------
# 4. Data setup for fine-tuning
# ---------------------------------------------------
train_transform = transforms.Compose([
    transforms.Resize((1024, 1024)),  # use 1024x1024 if GPU memory allows
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_data = datasets.ImageFolder(root="data/train", transform=train_transform)
train_loader = DataLoader(train_data, batch_size=16, shuffle=True, num_workers=2)

num_classes = len(train_data.classes)
print("Detected classes:", train_data.classes)

# ---------------------------------------------------
# 5. Add classification head
# ---------------------------------------------------
finetune_head = nn.Linear(128, num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    list(finetune_head.parameters()) +
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-4
)

# ---------------------------------------------------
# 6. Fine-tuning loop
# ---------------------------------------------------
epochs = 30
for epoch in range(epochs):
    model.train()
    finetune_head.train()
    running_loss = 0.0

    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()

        feats = model(imgs)
        outputs = finetune_head(feats)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/len(train_loader):.4f}")

torch.save(model.state_dict(), "hybrid_encoder_finetuned.pth")
torch.save(finetune_head.state_dict(), "hybrid_head.pth")
print("✅ Saved fine-tuned hybrid encoder and head.")

# ---------------------------------------------------
# 7. Extract fine-tuned features
# ---------------------------------------------------
model.eval()
finetune_head.eval()
all_features, all_labels = [], []

with torch.no_grad():
    for imgs, labels in tqdm(train_loader, desc="Extracting Fine-Tuned Features"):
        imgs = imgs.to(device)
        feats = model(imgs).cpu().numpy()
        all_features.append(feats)
        all_labels.append(labels.cpu().numpy())

X = np.concatenate(all_features)
y = np.concatenate(all_labels)

print("Feature matrix shape:", X.shape)
print("Label vector shape:", y.shape)

with open("hybrid_features_finetuned.pkl", "wb") as f:
    pickle.dump({"features": X, "labels": y}, f)

print("✅ Saved fine-tuned hybrid features to hybrid_features_finetuned.pkl")


Using device: cpu


RuntimeError: Error(s) in loading state_dict for HybridEncoder:
	size mismatch for projection.0.weight: copying a param with shape torch.Size([512, 512]) from checkpoint, the shape in current model is torch.Size([1024, 512]).
	size mismatch for projection.0.bias: copying a param with shape torch.Size([512]) from checkpoint, the shape in current model is torch.Size([1024]).
	size mismatch for projection.2.weight: copying a param with shape torch.Size([128, 512]) from checkpoint, the shape in current model is torch.Size([128, 1024]).